In [5]:
import re
import numpy as np
import pandas as pd

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [2]:
ANSI_RE = re.compile(
    r"#033\[[0-9;]*m|#015|#012|#011|\x1b\[[0-9;]*m"
)

SYSLOG_RE = re.compile(
    r"^(?P<syslog_ts>\d{4}-\d{2}-\d{2}T[^\s]+)\s+(?P<rest>.*)$"
)

APP_RE = re.compile(
    r"(?P<app_ts>\d{4}-\d{2}-\d{2}\s+\d{2}:\s?\d{2}:\s?\d{2},\d+)\s+"
    r"(?P<component>[A-Za-z0-9_\-\[\]]+)\s+"
    r"(?P<severity>INFO|WARNING|ERROR|CRITICAL):\s+"
    r"(?P<message>.*)"
)

CLASSIC_RE = re.compile(
    r"(?P<host>\S+)\s+"
    r"(?P<process>[A-Za-z0-9_\-]+)(?:\[(?P<pid>\d+)\])?:\s+"
    r"(?P<message>.*)"
)

In [3]:
def clean_message(text):
    if text is None:
        return ""

    text = ANSI_RE.sub(" ", text)
    text = text.replace("│", " ")
    text = text.replace("┌", " ").replace("┐", " ")
    text = text.replace("└", " ").replace("┘", " ")
    text = text.replace("├", " ").replace("┤", " ")
    text = text.replace("─", " ").replace("┬", " ")
    text = text.replace("┴", " ").replace("┼", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [4]:
def parse_line(line):
    line = clean_message(line)

    m = SYSLOG_RE.match(line)
    if not m:
        return None

    syslog_ts = m.group("syslog_ts")
    rest = m.group("rest")

    row = {
        "syslog_ts": syslog_ts,
        "app_ts": None,
        "host": None,
        "process": None,
        "component": None,
        "severity": "UNKNOWN",
        "message": rest,
        "device_id": None,
        "device_type": "other",
        "num_values": [],
    }

    app = APP_RE.search(rest)

    if app:
        row["app_ts"] = app.group("app_ts").replace(": ", ":")
        row["component"] = app.group("component")
        row["severity"] = app.group("severity")
        row["message"] = app.group("message")

    else:
        classic = CLASSIC_RE.match(rest)

        if classic:
            row["host"] = classic.group("host")
            row["process"] = classic.group("process")
            row["message"] = classic.group("message")

            sev = re.search(r"\[(INFO|WARNING|ERROR|CRITICAL)\]", row["message"], re.I)
            if sev:
                row["severity"] = sev.group(1).upper()

    row["message"] = clean_message(row["message"])

    combined_text = " ".join(
        str(row.get(k) or "") for k in ["component", "process", "message"]
    )

    dev = re.search(r"\b(PLC1|PLC2|HMI1|HMI2|HMI3)\b", combined_text, re.I)

    if dev:
        row["device_id"] = dev.group(1).upper()
        row["device_type"] = "PLC" if row["device_id"].startswith("PLC") else "HMI"
    elif "kernel" in combined_text.lower():
        row["device_type"] = "kernel"
    elif "rsyslog" in combined_text.lower():
        row["device_type"] = "syslog"
    elif "attacker" in combined_text.lower():
        row["device_type"] = "attacker"
    elif "snapshot" in combined_text.lower():
        row["device_type"] = "snapshot"

    nums = re.findall(r"[-+]?\d*\.\d+|[-+]?\d+", row["message"])
    row["num_values"] = [float(x) for x in nums]

    return row

In [ ]:
def parse_file(path, label_name, label_value):
    rows = []

    path = Path(path)

    for line in path.read_text(errors="ignore").splitlines():
        parsed = parse_line(line)

        if parsed is not None:
            parsed["label_name"] = label_name
            parsed["label"] = label_value
            parsed["source_file"] = path.name
            rows.append(parsed)

    df = pd.DataFrame(rows)

    df["syslog_ts"] = pd.to_datetime(df["syslog_ts"], errors="coerce", utc=True)
    df = df.dropna(subset=["syslog_ts"])

    return df


fault_df = parse_file("/home/ubuntu/raw_data/fault/all_logs.log", label_name="fault", label_value=1)
normal_df = parse_file("/home/ubuntu/raw_data/normal/all_logs.log", label_name="normal", label_value=0)

logs = pd.concat([normal_df, fault_df], ignore_index=True)

print(logs.head())
print(logs["label_name"].value_counts())

                         syslog_ts                   app_ts          host  \
0 2025-01-09 14:43:07.159018+00:00                     None  15b16d2baaa8   
1 2025-01-09 14:43:07.159020+00:00                     None  15b16d2baaa8   
2 2025-01-09 14:43:07.159021+00:00                     None  15b16d2baaa8   
3 2025-01-09 14:43:07.159022+00:00                     None  15b16d2baaa8   
4 2025-01-09 14:43:07.304726+00:00  2025-01-09 22:43:06,277          None   

    process  component severity  \
0  rsyslogd       None  UNKNOWN   
1  rsyslogd       None  UNKNOWN   
2  rsyslogd       None  UNKNOWN   
3  rsyslogd       None  UNKNOWN   
4      None  logs-HMI3     INFO   

                                             message device_id device_type  \
0  environment variable TZ is not set, auto corre...      None      syslog   
1                  rsyslogd's groupid changed to 102      None      syslog   
2                   rsyslogd's userid changed to 101      None      syslog   
3  [origin sof

In [6]:
KEYWORD_PATTERNS = {
    "kw_sensor_drift": r"sensor drift|drifted",
    "kw_tank_leak": r"tank leak|level reduced",
    "kw_valve_sticking": r"valve sticking|valve set to",
    "kw_conveyor_sticking": r"conveyor belt sticking|belt stopped",
    "kw_memory_corruption": r"memory corruption",
    "kw_overheating": r"overheating|processing delayed",
    "kw_null_value": r"null|nonetype|receive null value",
    "kw_latency": r"latency",
    "kw_fault_word": r"\bfault\b|\bfaults\b",
}

In [7]:
def template_message(msg):
    msg = str(msg).lower()
    msg = re.sub(r"192\.168\.\d+\.\d+:\d+", "<IP_PORT>", msg)
    msg = re.sub(r"\b\d+\.\d+\b", "<NUM>", msg)
    msg = re.sub(r"\b\d+\b", "<NUM>", msg)
    msg = re.sub(r"\s+", " ", msg)
    return msg.strip()


logs["template"] = logs["message"].apply(template_message)

In [8]:
def add_keyword_flags(df):
    df = df.copy()

    lower_msg = df["message"].fillna("").str.lower()

    for feature_name, pattern in KEYWORD_PATTERNS.items():
        df[feature_name] = lower_msg.str.contains(
            pattern,
            regex=True,
            case=False,
            na=False
        ).astype(int)

    return df


logs = add_keyword_flags(logs)

In [9]:
def summarize_numeric_values(values):
    nums = [x for sublist in values for x in sublist]

    if len(nums) == 0:
        return {
            "num_count": 0,
            "num_mean": 0,
            "num_std": 0,
            "num_min": 0,
            "num_max": 0,
            "num_range": 0,
        }

    nums = np.array(nums, dtype=float)

    return {
        "num_count": len(nums),
        "num_mean": nums.mean(),
        "num_std": nums.std(),
        "num_min": nums.min(),
        "num_max": nums.max(),
        "num_range": nums.max() - nums.min(),
    }


def make_windows(df, window_size="5s"):
    df = df.copy()
    df = df.sort_values("syslog_ts")
    df = df.set_index("syslog_ts")

    rows = []

    keyword_cols = list(KEYWORD_PATTERNS.keys())

    for label_value, label_group in df.groupby("label"):
        label_name = label_group["label_name"].iloc[0]

        for window_start, win in label_group.resample(window_size):
            if len(win) == 0:
                continue

            severities = win["severity"].fillna("UNKNOWN").str.upper()
            device_ids = win["device_id"].fillna("UNKNOWN")
            device_types = win["device_type"].fillna("other")
            messages = win["message"].fillna("")

            row = {
                "window_start": window_start,
                "label": label_value,
                "label_name": label_name,

                # General count features
                "log_count": len(win),
                "unique_components": win["component"].nunique(),
                "unique_processes": win["process"].nunique(),
                "unique_devices": win["device_id"].nunique(),
                "unique_templates": win["template"].nunique(),

                # Severity features
                "info_count": (severities == "INFO").sum(),
                "warning_count": (severities == "WARNING").sum(),
                "error_count": (severities == "ERROR").sum(),
                "critical_count": (severities == "CRITICAL").sum(),
                "unknown_severity_count": (severities == "UNKNOWN").sum(),

                # Device ID features
                "plc1_count": (device_ids == "PLC1").sum(),
                "plc2_count": (device_ids == "PLC2").sum(),
                "hmi1_count": (device_ids == "HMI1").sum(),
                "hmi2_count": (device_ids == "HMI2").sum(),
                "hmi3_count": (device_ids == "HMI3").sum(),

                # Device type features
                "plc_count": (device_types == "PLC").sum(),
                "hmi_count": (device_types == "HMI").sum(),
                "kernel_count": (device_types == "kernel").sum(),
                "syslog_count": (device_types == "syslog").sum(),
                "snapshot_count": (
                    win["component"].fillna("").str.contains("snapshot", case=False).sum()
                ),

                # Message shape features
                "mean_msg_len": messages.str.len().mean(),
                "max_msg_len": messages.str.len().max(),
                "min_msg_len": messages.str.len().min(),
                "repeated_template_count": len(win) - win["template"].nunique(),
            }

            # Ratios
            row["warning_ratio"] = row["warning_count"] / row["log_count"]
            row["error_ratio"] = row["error_count"] / row["log_count"]
            row["critical_ratio"] = row["critical_count"] / row["log_count"]
            row["plc_ratio"] = row["plc_count"] / row["log_count"]
            row["hmi_ratio"] = row["hmi_count"] / row["log_count"]

            # Numeric embedded values
            row.update(summarize_numeric_values(win["num_values"]))

            # Keyword features
            for col in keyword_cols:
                row[col + "_count"] = win[col].sum()
                row[col + "_ratio"] = win[col].sum() / row["log_count"]

            rows.append(row)

    windows = pd.DataFrame(rows)

    return windows


features = make_windows(logs, window_size="5s")

print(features.head())
print(features["label_name"].value_counts())

               window_start  label label_name  log_count  unique_components  \
0 2025-01-09 14:43:05+00:00      0     normal        111                 10   
1 2025-01-09 14:43:10+00:00      0     normal        145                  4   
2 2025-01-09 14:43:15+00:00      0     normal        145                  4   
3 2025-01-09 14:43:20+00:00      0     normal        145                  4   
4 2025-01-09 14:43:25+00:00      0     normal        145                  4   

   unique_processes  unique_devices  unique_templates  info_count  \
0                 2               5                41          93   
1                 0               3                15         145   
2                 0               3                15         145   
3                 0               3                17         145   
4                 0               3                17         145   

   warning_count  ...  kw_memory_corruption_count  kw_memory_corruption_ratio  \
0             13  ...        

In [10]:
target_col = "label"

metadata_cols = [
    "window_start",
    "label",
    "label_name",
]

keyword_feature_cols = [
    col for col in features.columns
    if col.startswith("kw_")
]

all_feature_cols = [
    col for col in features.columns
    if col not in metadata_cols
]

non_keyword_feature_cols = [
    col for col in all_feature_cols
    if col not in keyword_feature_cols
]

print("Non-keyword features:", non_keyword_feature_cols)
print("Keyword features:", keyword_feature_cols)

Non-keyword features: ['log_count', 'unique_components', 'unique_processes', 'unique_devices', 'unique_templates', 'info_count', 'warning_count', 'error_count', 'critical_count', 'unknown_severity_count', 'plc1_count', 'plc2_count', 'hmi1_count', 'hmi2_count', 'hmi3_count', 'plc_count', 'hmi_count', 'kernel_count', 'syslog_count', 'snapshot_count', 'mean_msg_len', 'max_msg_len', 'min_msg_len', 'repeated_template_count', 'warning_ratio', 'error_ratio', 'critical_ratio', 'plc_ratio', 'hmi_ratio', 'num_count', 'num_mean', 'num_std', 'num_min', 'num_max', 'num_range']
Keyword features: ['kw_sensor_drift_count', 'kw_sensor_drift_ratio', 'kw_tank_leak_count', 'kw_tank_leak_ratio', 'kw_valve_sticking_count', 'kw_valve_sticking_ratio', 'kw_conveyor_sticking_count', 'kw_conveyor_sticking_ratio', 'kw_memory_corruption_count', 'kw_memory_corruption_ratio', 'kw_overheating_count', 'kw_overheating_ratio', 'kw_null_value_count', 'kw_null_value_ratio', 'kw_latency_count', 'kw_latency_ratio', 'kw_faul

In [11]:
X_no_keywords = features[non_keyword_feature_cols]
X_with_keywords = features[all_feature_cols]
y = features[target_col]

X_train_no, X_test_no, y_train, y_test = train_test_split(
    X_no_keywords,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

X_train_kw, X_test_kw, _, _ = train_test_split(
    X_with_keywords,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [12]:
model_no_keywords = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    max_depth=None
)

model_no_keywords.fit(X_train_no, y_train)

pred_no = model_no_keywords.predict(X_test_no)
prob_no = model_no_keywords.predict_proba(X_test_no)[:, 1]

print("=== Model WITHOUT keyword features ===")
print("Accuracy:", accuracy_score(y_test, pred_no))
print("ROC-AUC:", roc_auc_score(y_test, prob_no))
print(confusion_matrix(y_test, pred_no))
print(classification_report(y_test, pred_no, target_names=["normal", "fault"]))

=== Model WITHOUT keyword features ===
Accuracy: 1.0
ROC-AUC: 1.0
[[984   0]
 [  0 571]]
              precision    recall  f1-score   support

      normal       1.00      1.00      1.00       984
       fault       1.00      1.00      1.00       571

    accuracy                           1.00      1555
   macro avg       1.00      1.00      1.00      1555
weighted avg       1.00      1.00      1.00      1555



In [13]:
model_with_keywords = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    max_depth=None
)

model_with_keywords.fit(X_train_kw, y_train)

pred_kw = model_with_keywords.predict(X_test_kw)
prob_kw = model_with_keywords.predict_proba(X_test_kw)[:, 1]

print("=== Model WITH keyword features ===")
print("Accuracy:", accuracy_score(y_test, pred_kw))
print("ROC-AUC:", roc_auc_score(y_test, prob_kw))
print(confusion_matrix(y_test, pred_kw))
print(classification_report(y_test, pred_kw, target_names=["normal", "fault"]))

=== Model WITH keyword features ===
Accuracy: 1.0
ROC-AUC: 1.0
[[984   0]
 [  0 571]]
              precision    recall  f1-score   support

      normal       1.00      1.00      1.00       984
       fault       1.00      1.00      1.00       571

    accuracy                           1.00      1555
   macro avg       1.00      1.00      1.00      1555
weighted avg       1.00      1.00      1.00      1555



In [14]:
def show_feature_importance(model, feature_cols, top_n=20):
    importances = pd.DataFrame({
        "feature": feature_cols,
        "importance": model.feature_importances_
    })

    importances = importances.sort_values("importance", ascending=False)

    return importances.head(top_n)


importance_no_keywords = show_feature_importance(
    model_no_keywords,
    non_keyword_feature_cols,
    top_n=20
)

importance_with_keywords = show_feature_importance(
    model_with_keywords,
    all_feature_cols,
    top_n=20
)

print("Top features WITHOUT keywords:")
print(importance_no_keywords)

print("\nTop features WITH keywords:")
print(importance_with_keywords)

Top features WITHOUT keywords:
                    feature  importance
29                num_count    0.073333
28                hmi_ratio    0.070008
5                info_count    0.070000
34                num_range    0.069862
0                 log_count    0.066667
33                  num_max    0.066667
11               plc2_count    0.066667
8            critical_count    0.063389
26           critical_ratio    0.060000
27                plc_ratio    0.056671
20             mean_msg_len    0.056667
23  repeated_template_count    0.053333
15                plc_count    0.050000
19           snapshot_count    0.046281
24            warning_ratio    0.039997
10               plc1_count    0.030000
7               error_count    0.026652
25              error_ratio    0.019995
6             warning_count    0.013333
21              max_msg_len    0.000335

Top features WITH keywords:
                       feature  importance
50            kw_latency_ratio    0.066667
0             

In [15]:
logreg_no_keywords = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

logreg_no_keywords.fit(X_train_no, y_train)

pred_lr_no = logreg_no_keywords.predict(X_test_no)
prob_lr_no = logreg_no_keywords.predict_proba(X_test_no)[:, 1]

print("=== Logistic Regression WITHOUT keyword features ===")
print("Accuracy:", accuracy_score(y_test, pred_lr_no))
print("ROC-AUC:", roc_auc_score(y_test, prob_lr_no))
print(confusion_matrix(y_test, pred_lr_no))
print(classification_report(y_test, pred_lr_no, target_names=["normal", "fault"]))

=== Logistic Regression WITHOUT keyword features ===
Accuracy: 1.0
ROC-AUC: 1.0
[[984   0]
 [  0 571]]
              precision    recall  f1-score   support

      normal       1.00      1.00      1.00       984
       fault       1.00      1.00      1.00       571

    accuracy                           1.00      1555
   macro avg       1.00      1.00      1.00      1555
weighted avg       1.00      1.00      1.00      1555



In [16]:
features_sorted = features.sort_values("window_start").reset_index(drop=True)

split_idx = int(len(features_sorted) * 0.7)

train_df = features_sorted.iloc[:split_idx]
test_df = features_sorted.iloc[split_idx:]

X_train_no = train_df[non_keyword_feature_cols]
X_test_no = test_df[non_keyword_feature_cols]

X_train_kw = train_df[all_feature_cols]
X_test_kw = test_df[all_feature_cols]

y_train = train_df["label"]
y_test = test_df["label"]

In [17]:
logs.to_csv("parsed_syslogs.csv", index=False)
features.to_csv("syslog_window_features.csv", index=False)